<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/1_bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Bit LLM

In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [8]:
class BitLinear(nn.Linear):
  def forward(self, x: torch.Tensor):
    # Scale input tensor down to 8 bits
    i_scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
    x_quant = torch.round(x * 127 / i_scale).clamp(-128, 127)

    # Scale layer weights to ternary -,+,0
    w = self.weight
    w_scale = w.abs().mean()
    w_quant = torch.round(w / (w_scale + 1e-5)).clamp(-1, 1)
    w_ternary = w + (w_quant - w).detach()

    out_t = F.linear(x_quant, w_ternary, self.bias)

    # Scale output tensor up to 8bits
    out_t = out_t * (i_scale / 127) * (w_scale)
    return out_t

Tests for the BitLinear class

In [10]:
bit_linear_layer = BitLinear(10, 5)
x = torch.randn(1, 10)
output = bit_linear_layer(x)

assert isinstance(output, torch.Tensor), "Assertion Failed: Output should be a torch.Tensor"
assert output.shape == (1, 5), f"Assertion Failed: Output shape mismatch. Expected (1, {5}), got {output.shape}"
assert not torch.isnan(output).any(), "Assertion Failed: Output contains NaN values"
assert not torch.isinf(output).any(), "Assertion Failed: Output contains Inf values"
print(f"Output tensor (first 5 elements): {output.flatten()[:5].tolist()}")

Output tensor (first 5 elements): [-0.3493382930755615, -0.47196802496910095, 0.30982455611228943, 0.6833235025405884, -0.2877483069896698]
